# Training a lead sheet language model

This notebook trains the language model that generates lead sheets in Chapter 16 of
_Principles of Computer Music_. The model is a small _recurrent neural network_ (RNN),
trained on about $9{,}000$ lead sheets (melodies and chords) of pop songs from the
Hooktheory dataset. It then saves the trained parameters to
`../assets/lead-sheet-rnn.npz`, which the chapter loads to generate new lead sheets.

Unlike the code embedded in the book, this notebook needs [PyTorch](https://pytorch.org),
so it does not run in the browser. Training takes about 25 minutes on a laptop CPU.

In [ ]:
import gzip
import json

import numpy as np
import torch
import torch.nn.functional as F

## Tokens

A language model works with sequences of _tokens_ from a finite vocabulary. We encode
each lead sheet as a sequence of events, in time order:

- `BAR` starts a new bar (four beats),
- `CHORD_<root>:<type>` starts a new chord, for example `CHORD_F#:min7`,
- `NOTE_<pitch>` plays a melody note, a MIDI pitch from $48$ (C3) to $84$ (C6),
- `SHIFT_<k>` waits $k$ sixteenth notes (a sixteenth of a bar is a quarter of a beat),

plus `<BOS>` and `<EOS>` tokens that mark the beginning and end of a song.

In [ ]:
PITCH_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
CHORD_TYPES = ["maj", "min", "7", "maj7", "min7"]
LOWEST, HIGHEST = 48, 84                         # the range of melody pitches, C3 to C6
VOCAB = (["<BOS>", "<EOS>", "BAR"]
         + [f"CHORD_{root}:{kind}" for root in PITCH_NAMES for kind in CHORD_TYPES]
         + [f"NOTE_{pitch}" for pitch in range(LOWEST, HIGHEST + 1)]
         + [f"SHIFT_{k}" for k in range(1, 16)])
INDEX = {token: i for i, token in enumerate(VOCAB)}


def events_to_tokens(melody: list, chords: list) -> list[int]:
    """A lead sheet as token indices: <BOS>, then each bar as a BAR token followed by its
    chords and notes in time order (a chord before a note at the same time), with SHIFT
    tokens in between, then <EOS>. Onsets are rounded to sixteenth notes, and pitches are
    moved by octaves into range."""
    events = {}
    for beat, label in chords:
        events[round(4 * beat), 0] = f"CHORD_{label}"
    for beat, pitch in melody:
        while pitch < LOWEST:
            pitch += 12
        while pitch > HIGHEST:
            pitch -= 12
        events[round(4 * beat), 1] = f"NOTE_{pitch}"
    tokens, bar, position = ["<BOS>"], -1, 0
    for (sixteenth, _), token in sorted(events.items()):
        while bar < sixteenth // 16:              # start a new bar
            tokens.append("BAR")
            bar, position = bar + 1, 0
        if position < sixteenth % 16:             # wait until this event
            tokens.append(f"SHIFT_{sixteenth % 16 - position}")
            position = sixteenth % 16
        tokens.append(token)
    tokens.append("<EOS>")
    return [INDEX[token] for token in tokens]


with gzip.open("../assets/TheorytabSimple.json.gz", "rt") as f:
    dataset = json.load(f)
songs = {"TRAIN": [], "VALID": [], "TEST": []}
for song in dataset.values():
    songs[song["split"]].append(events_to_tokens(song["melody"], song["chords"]))
for split in songs:
    print(f"{split}: {len(songs[split])} songs, {sum(map(len, songs[split]))} tokens")
print([VOCAB[token] for token in songs["TRAIN"][0][:12]], "...")

## The model

The RNN reads one token at a time. It summarizes everything that it has read so far in
a _hidden state_ $\mathbf{h}_t$, a vector of $256$ numbers, which it updates with each
new token $x_t$:

$$\mathbf{h}_t = \tanh\left(\mathbf{E}[x_t] \, \mathbf{W}_x + \mathbf{h}_{t-1} \mathbf{W}_h + \mathbf{b}_h\right).$$

Here, $\mathbf{E}[x_t]$ is the _embedding_ of the token, a learned vector of $32$
numbers. From the hidden state, the model predicts the probability of each possible
next token, with a matrix multiplication followed by a _softmax_, which turns a vector
of real numbers into probabilities that sum to one:

$$P_\theta(x_{t+1} \mid x_1, \ldots, x_t) = \text{softmax}\left(\mathbf{h}_t \mathbf{W}_y + \mathbf{b}_y\right).$$

The parameters $\theta$ are $\mathbf{E}$, $\mathbf{W}_x$, $\mathbf{W}_h$,
$\mathbf{b}_h$, $\mathbf{W}_y$, and $\mathbf{b}_y$. PyTorch provides each piece.

In [ ]:
class LeadSheetRNN(torch.nn.Module):
    def __init__(self, embed_dim: int = 32, hidden_dim: int = 256):
        super().__init__()
        self.embed = torch.nn.Embedding(len(VOCAB), embed_dim)
        self.rnn = torch.nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.out = torch.nn.Linear(hidden_dim, len(VOCAB))

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """Tokens (batch, time) -> unnormalized log-probabilities of the next tokens
        (batch, time, vocab). The softmax is part of the loss function."""
        h, _ = self.rnn(self.embed(tokens))
        return self.out(h)


def bits_per_token(model: LeadSheetRNN, songs: list[list[int]]) -> float:
    """The average negative log-likelihood (in bits) of each token after <BOS>, over
    whole songs."""
    total, count = 0.0, 0
    with torch.no_grad():
        for song in songs:
            tokens = torch.tensor(song)[None]
            logits = model(tokens[:, :-1])
            total += F.cross_entropy(logits[0], tokens[0, 1:], reduction="sum").item()
            count += len(song) - 1
    return total / count / np.log(2)


torch.manual_seed(0)
model = LeadSheetRNN()
print(f"{sum(p.numel() for p in model.parameters())} parameters")
print(f"Before training: {bits_per_token(model, songs['VALID']):.2f} bits per token")

## Training

We minimize the average negative log-likelihood of the next token (the _cross-entropy_
loss) on random excerpts of $128$ tokens from the training songs, $64$ at a time, with
the Adam optimizer, a variant of gradient descent. PyTorch computes the gradients for us
with backpropagation (`loss.backward()`).

To get more out of the data, we _transpose_ each excerpt by a random number of
semitones (moving all of its notes and chords up or down by the same amount), so that
the model sees every song in all $12$ keys. Without this, the model starts to memorize
the training songs (to _overfit_).

Every $2{,}000$ steps, we measure the loss on the held-out VALID songs, and keep the
parameters that do best.

In [ ]:
def transposition_table() -> np.ndarray:
    """table[s + 6, token] is the token transposed by s = -6, ..., 5 semitones. Notes
    move by s (and by octaves, to stay in range), chord roots move by s, and every other
    token stays the same."""
    table = np.zeros((12, len(VOCAB)), dtype=np.int64)
    for s in range(-6, 6):
        for i, token in enumerate(VOCAB):
            kind, _, value = token.partition("_")
            if kind == "NOTE":
                pitch = int(value) + s
                token = f"NOTE_{pitch + 12 * (pitch < LOWEST) - 12 * (pitch > HIGHEST)}"
            elif kind == "CHORD":
                root, chord_type = value.split(":")
                token = f"CHORD_{PITCH_NAMES[(PITCH_NAMES.index(root) + s) % 12]}:{chord_type}"
            table[s + 6, i] = INDEX[token]
    return table


STEPS, BATCH_SIZE, LENGTH = 60_000, 64, 128
rng = np.random.default_rng(0)
transpose = transposition_table()
stream = np.concatenate(songs["TRAIN"])          # all of the training songs, end to end
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, STEPS, eta_min=5e-5)
best = np.inf
for step in range(1, STEPS + 1):
    starts = rng.integers(0, len(stream) - LENGTH - 1, size=(BATCH_SIZE, 1))
    excerpts = stream[starts + np.arange(LENGTH + 1)]                # (batch, time + 1)
    excerpts = torch.from_numpy(transpose[rng.integers(0, 12, size=(BATCH_SIZE, 1)), excerpts])
    logits = model(excerpts[:, :-1])                                 # predict each next token
    loss = F.cross_entropy(logits.reshape(-1, len(VOCAB)), excerpts[:, 1:].reshape(-1))
    optimizer.zero_grad()
    loss.backward()                                                  # backpropagation
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)          # avoid huge steps
    optimizer.step()                                                 # gradient descent
    schedule.step()                                                  # lower the learning rate
    if step % 2000 == 0:
        valid = bits_per_token(model, songs["VALID"])
        if valid < best:
            best, best_state = valid, {k: v.clone() for k, v in model.state_dict().items()}
        print(f"step {step:5d}: training loss {loss.item() / np.log(2):.2f}, "
              f"VALID {valid:.3f} bits per token")
model.load_state_dict(best_state)
print(f"Best: VALID {bits_per_token(model, songs['VALID']):.3f}, "
      f"TEST {bits_per_token(model, songs['TEST']):.3f} bits per token")

## Saving the model

Finally, we save the parameters as NumPy arrays, with the names in the equations
above, so that the book can run the model with NumPy alone. (PyTorch stores its
weight matrices transposed, and splits $\mathbf{b}_h$ into two biases.)

In [ ]:
weights = {name: w.numpy() for name, w in model.state_dict().items()}
np.savez("../assets/lead-sheet-rnn.npz", vocab=np.array(VOCAB),
         valid_bits_per_token=np.float32(best),
         E=weights["embed.weight"],
         W_x=weights["rnn.weight_ih_l0"].T, W_h=weights["rnn.weight_hh_l0"].T,
         b_h=weights["rnn.bias_ih_l0"] + weights["rnn.bias_hh_l0"],
         W_y=weights["out.weight"].T, b_y=weights["out.bias"])